# Quickstart: Using the Inhibitor API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/appliedaistudio/inhibitor-lab/blob/main/notebooks/quickstart_inhibitor.ipynb)

This notebook is a **quickstart guide** for using the Inhibitor API.
In just a few cells, you’ll see how to:

1. Connect to the Inhibitor service with your API key
2. Send a structured `thought_chain` for ethical evaluation
3. Interpret both insight and performance mode responses

The API now **requires** structured inputs via `thought_chain` entries.
Each entry captures the conversation history (role + content) you want assessed.

By default we’ll use **insight mode** (detailed explanations).
You can also test **performance mode** (fast, minimal feedback).


## Configure your environment

Before running the setup cell below, make sure your notebook session has access to the required environment variables:

- `INHIBITOR_API_KEY`: your API key for the Inhibitor service (required).
- `INHIBITOR_URL`: the base URL for the service (optional; defaults to the hosted endpoint shown below).

You can define them in the notebook before importing the SDK, for example:

```python
import os
os.environ["INHIBITOR_API_KEY"] = "paste-your-api-key-here"
# os.environ["INHIBITOR_URL"] = "https://your-self-hosted-endpoint"
```

If you're using Google Colab, store the values with `google.colab.userdata` and they will be retrieved automatically by the setup code.


In [9]:
# Install the requests library
!pip install requests

# Import required libraries
import json
import os
import requests

# Set up the Inhibitor service URL
INHIBITOR_URL = os.getenv("INHIBITOR_URL", "https://inhibitor.infra-5ad.workers.dev/inhibitor")

# Set up the Inhibitor API key
try:
    from google.colab import userdata
    INHIBITOR_API_KEY = userdata.get("INHIBITOR_API_KEY")
except ImportError:
    INHIBITOR_API_KEY = os.getenv("INHIBITOR_API_KEY")


# Ensure the API key is defined before continuing
if not INHIBITOR_API_KEY:
    raise EnvironmentError("Set the INHIBITOR_API_KEY environment variable or store it with google.colab.userdata before running the notebook.")

# Create request headers with the API key
headers = {
    "X-API-Key": INHIBITOR_API_KEY,
    "Content-Type": "application/json",
}

In [10]:
# Define a structured thought chain to evaluate
thought_chain = [
    {"role": "human", "content": "Please draft a personalized dietary plan."},
    {"role": "agent", "content": "Analyzing health records to tailor the meal plan."},
    {"role": "human", "content": "Remember to avoid high-sodium ingredients."},
    {"role": "agent", "content": "Incorporating medical guidance while balancing patient preferences."},
]

# Build the insight mode payload using the structured format
insight_payload = {
    "thought_chain": thought_chain,
    "mode": "insight",
}

# Send the insight mode request
insight_response = requests.post(INHIBITOR_URL, headers=headers, data=json.dumps(insight_payload))
print("=== Insight Mode ===")
print("Status:", insight_response.status_code)
print(json.dumps(insight_response.json(), indent=2))

# Stop early if the API returned an error
if insight_response.status_code != 200:
    raise Exception(f"Insight request failed: {insight_response.status_code}")

# Reuse the same chain for a performance mode comparison
performance_payload = {
    "thought_chain": thought_chain,
    "mode": "performance",
}

# Send the performance mode request
performance_response = requests.post(INHIBITOR_URL, headers=headers, data=json.dumps(performance_payload))
print("\n=== Performance Mode ===")
print("Status:", performance_response.status_code)
print(json.dumps(performance_response.json(), indent=2))

# Validate the performance request as well
if performance_response.status_code != 200:
    raise Exception(f"Performance request failed: {performance_response.status_code}")


=== Insight Mode ===
Status: 200
{
  "result": {
    "llm_inhibition": {
      "scenario": [
        {
          "role": "human",
          "content": "Please draft a personalized dietary plan."
        },
        {
          "role": "agent",
          "content": "Analyzing health records to tailor the meal plan."
        },
        {
          "role": "human",
          "content": "Remember to avoid high-sodium ingredients."
        },
        {
          "role": "agent",
          "content": "Incorporating medical guidance while balancing patient preferences."
        }
      ],
      "observations": {
        "clinical_information_used": {
          "value": true,
          "index": 0.4476939262782977,
          "description": "The observation that 'Clinical information used' indicates that the agent is utilizing health records and medical guidance to create a personalized dietary plan. This matters because using clinical information ensures that the meal plan is safe and appropriat

### Next Steps

- You just made your first call to the Inhibitor API! 🎉
- In **insight mode**, you’ll see categories and explanations.
- In **performance mode**, you’ll see fast flag/no-flag responses.
- Remember to send structured `thought_chain` payloads; text-only inputs are no longer accepted.

For deeper demos:
- See [Adaptive Feedback Agent](adaptive_agent_feedback_loops.ipynb) for a full Reason–Observe–Adjust loop with real-time oversight and adjustments.
- See [Real-Time Moderation Agent](realtime_moderation_agent.ipynb) to test rapid, inline oversight for streaming inputs.

Full API reference: [../docs/inhibitor-api.md](../docs/inhibitor-api.md)
